Data ingestion pipeline - from Ingestion to VectorDB

In [1]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
#from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/home/zr/anaconda3/envs/rag311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def process_pdf(pdf_dir):
    all_docs = []
    pdf_dir = Path(pdf_dir)

    pdf_files = list(pdf_dir.glob("*.pdf"))
    print(f"found {len(pdf_files)} in pdf directory to process")

    for pdf_file in pdf_files:
        print(f"processing {pdf_file.name}")
        loader = PyMuPDFLoader(str(pdf_file))
        #loader = PyPDFLoader(str(pdf_file))
        docs = loader.load()
        for doc in docs:
            doc.metadata = {"source": pdf_file.name}
            doc.metadata = {"file_type": "pdf"}
        all_docs.extend(docs)
        print(f"loaded {len(docs)} pages from {pdf_file.name}")

    print(f"total docs loaded so far: {len(all_docs)}")
    return all_docs

all_pdf_docs = process_pdf("../data/pdf")

found 3 in pdf directory to process
processing dl-notes.pdf
loaded 58 pages from dl-notes.pdf
processing ml-notes.pdf
loaded 278 pages from ml-notes.pdf
processing agentic-ai-notes.pdf
loaded 28 pages from agentic-ai-notes.pdf
total docs loaded so far: 364


In [3]:
def split_documents(docs, chunk_size=800, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size, 
        chunk_overlap = chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(docs)
    print(f"split {len(docs)} documents into {len(split_docs)} chunks")
    
    return split_docs


In [4]:
chunks = split_documents(all_pdf_docs)
#chunks

split 364 documents into 1117 chunks


Embeddings and VectorStoreDB

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer# embedding model
import uuid# id's for each chunk stored in VdB
import chromadb # vector database , faiss is another option for this
from chromadb.config import Settings
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        # This particular model is available on HuggingFace.
        # It converts chunks of text into embeddings(Vectors). 
        
        # Initialize the embedding manager
        
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
             print(f"Error loading model {self.model_name}: {e}")
             raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        #print("entered the generate_embeddings function")
        """
        Generate embeddings for a list of texts
        Arguments:
            texts: List of text strings to embed 
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        #if not self.model:
        #    raise ValueError("Model not loaded")
        #print("check the function run")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension: 384


In [7]:
## Vector Database 
class VectorStore():
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        # Initialize the vector store
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        # init the chromadb client
        #try:
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        # create or get the collection
        self.collection = self.client.get_or_create_collection(name=self.collection_name,
                                                                metadata={"description": "Pdf Document embeddings"})

        print("vector store initialized successfully.")
        print("existing documents in the collection:", self.collection.count())
        #except Exception as e:
        #    print("error initializing vector store:", e)
        #    raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        # Add documents and their embeddings to the vector store
        if len(documents) != len(embeddings):# check their lengths match
            raise ValueError("Number of documents and embeddings must match")

        print("adding documents to the vector store")
        ids =[]
        metadatas = []
        embeddings_list = []
        documents_text = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        self.collection.add(
            ids=ids,
            metadatas=metadatas,
            documents=documents_text,
            embeddings=embeddings_list
        )

        print("successfully added docs to Vector store")
        print("total docs in collection:", self.collection.count())

    def query(self, query_embeddings, n_results=5):
        return self.collection.query(
            query_embeddings=query_embeddings,
            n_results=n_results
            )

           
vectorstore=VectorStore()
vectorstore

vector store initialized successfully.
existing documents in the collection: 3351


In [8]:
# chunk text to embeddings and store in vector database

texts = [doc.page_content for doc in chunks]

# generate embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

# store the embeddings in the vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 1117 texts...


Batches: 100%|██████████| 35/35 [00:41<00:00,  1.19s/it]


Generated embeddings with shape: (1117, 384)
adding documents to the vector store
successfully added docs to Vector store
total docs in collection: 4468


In [9]:
## retriever class
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        # init the retriever with the vector store and embedding manager
        # why the 2 args: we need to convert the user query text into
        # embeddings and then search the vector store for similar embeddings

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve (self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        # retrieve relevant documents vec store for a given query
        #
        # Args:
        # query: user query string
        # top_k: number of top documents to return
        # score_threshold: minimum similarity score to consider a document relevant

        # Returns:
        # List of dictionaries with retrieved docs and metadata
        print("retrieving relevant content for your query")
        print(f"top_k : {top_k}, score threshold {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # search in store
        #try:
        results = self.vector_store.query(
            query_embeddings = [query_embedding.tolist()],
            n_results = top_k
        )

        retrieved_docs = []
        if results['documents'] and results['documents'][0]:
            documents = results['documents'][0]
            metadatas = results['metadatas'][0]
            distances = results['distances'][0]
            ids = results['ids'][0]

            for i, (doc, metadata, distance, doc_id) in enumerate(zip(documents, metadatas, distances, ids)):
                similarity_score = 1 - distance
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "content": doc,
                        "metadata": metadata,
                        "similarity_score": similarity_score,
                        "distance": distance,
                        "rank": i+1
                    })
            print(f"retrieved {len(retrieved_docs)} documents")
        else:
            print("no documents retrieved")

        return retrieved_docs
    #except Exception as e:
            #    print("error during retrieval:", e)
            #    raise

rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [10]:
rag_retriever.retrieve("what is sigmoid function?")

retrieving relevant content for your query
top_k : 5, score threshold 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.91it/s]

Generated embeddings with shape: (1, 384)
retrieved 4 documents


[{'id': 'doc_cba923d3_392',
  'content': 'less and less used these days as standalone hidden-layer activations partly be-\ncause they are bounded from both sides and their gradients vanish as z goes\nto both positive and negative inﬁnity (whereas all the other activation func-\ntions above still have gradients as the input goes to positive inﬁnity.) Sigmoid\nnevertheless remains important as a gating nonlinearity, for example in some\nmixture-of-experts routers [Nguyen et al., 2025] and gated attention mech-\nanisms [Qiu et al., 2025]. Softplus is not used very often in practice either\nand can be viewed as a smoothing of ReLU so that it has a proper second-\norder derivative. Swishβ was introduced by Ramachandran et al. [2017]; it',
  'metadata': {'file_type': 'pdf', 'content_length': 691, 'doc_index': 392},
  'similarity_score': 0.04153412580490112,
  'distance': 0.9584658741950989,
  'rank': 1},
 {'id': 'doc_e3f263a5_392',
  'content': 'less and less used these days as standalone hi

In [11]:
rag_retriever.retrieve("what is relu function?")

retrieving relevant content for your query
top_k : 5, score threshold 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 112.35it/s]

Generated embeddings with shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_d180ea26_393',
  'content': '91\nFigure 7.3: Activation functions in deep learning. The y-axis is capped above\n5 for visualization.\nis also commonly called SiLU, especially in the case β = 1, and is used in\narchitectures such as EﬃcientNet [Tan and Le, 2019]. GELU was introduced\nby Hendrycks and Gimpel [2016] and is widely used in Transformer language\nmodels such as BERT [Devlin et al., 2019], as well as in diﬀusion Transform-\ners such as Hunyuan-DiT [Li et al., 2024]. ReLU2 is a simple higher-order\nvariant of ReLU that is used in Primer [So et al., 2021] and was later found\nto improve sparsity in sparse LLMs [Zhang et al., 2024].\nAnother practically important family, especially in modern sequence mod-\nels, is gated activations. A gated linear unit (GLU) takes two aﬃne trans-',
  'metadata': {'file_type': 'pdf', 'content_length': 756, 'doc_index': 393},
  'similarity_score': 0.0738019347190857,
  'distance': 0.9261980652809143,
  'rank': 1},
 {'id': 'doc_0a77abaf

## Use llm with this pipeline for refined results from documents.

In [12]:
# The docs we retrieved + a user prompt become the input to the LLM for generating an answer.
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

from langchain_groq import ChatGroq
from langchain.prompts import PromptTemplate
from langchain.schema import SystemMessage, HumanMessage


In [13]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
            """
            Generate response using retrieved context
            
            Args:
                query: User question
                context: Retrieved document context
                max_length: Maximum response length
                
            Returns:
                Generated response string
            """
            
            # Create prompt template
            prompt_template = PromptTemplate(
                input_variables=["context", "question"],
                template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.
    
    Context:
    {context}
    
    Question: {question}
    
    Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
            )
            
            # Format the prompt
            formatted_prompt = prompt_template.format(context=context, question=query)
            
            try:
                # Generate response
                messages = [HumanMessage(content=formatted_prompt)]
                response = self.llm.invoke(messages)
                return response.content
                
            except Exception as e:
                return f"Error generating response: {str(e)}"
            
    def generate_response_simple(self, query: str, context: str) -> str:
            """
            Simple response generation without complex prompting
            
            Args:
                query: User question
                context: Retrieved context
                
            Returns:
                Generated response
            """
            simple_prompt = f"""Based on this context: {context}
    
    Question: {query}
    
    Answer:"""
            
            try:
                messages = [HumanMessage(content=simple_prompt)]
                response = self.llm.invoke(messages)
                return response.content
            except Exception as e:
                return f"Error: {str(e)}"

In [14]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [15]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="gemma2-9b-it",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [16]:
answer=rag_simple("What is attention mechanism?",rag_retriever,llm)
print(answer)

retrieving relevant content for your query
top_k : 3, score threshold 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 100.75it/s]

Generated embeddings with shape: (1, 384)
retrieved 3 documents


AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}